In [1]:
import os
import sys

try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    print("Entorno Colab detectado. Montando Google Drive...")
    drive.mount('/content/drive')
    # Ajustar la ruta según lo que se tenga en Google Drive
    ruta_proyecto = '/content/drive/MyDrive/RetinaScan-UNet' 
    if os.path.exists(ruta_proyecto):
        os.chdir(ruta_proyecto)
else:
    print("Entorno local detectado.")
    if os.path.basename(os.getcwd()) == "notebooks":
        os.chdir("..")

print("Directorio de trabajo actual:", os.getcwd())

Entorno local detectado.
Directorio de trabajo actual: C:\Users\maarc\Desktop\TERCERO_SOFTWARE\SEGUNDO CUATRI\IA\PROYECTO\retina_workspace\RetinaScan-UNet


In [2]:
# LIBRERÍAS
import re
import glob
import random
import numpy as np
import cv2
import matplotlib.pyplot as plt
import tensorflow as tf
import keras
from tensorflow.keras import layers, models
from keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau
from keras.optimizers import Adam
from sklearn.model_selection import KFold
from sklearn.metrics import precision_score, recall_score

In [3]:
# MÉTRICAS
@keras.saving.register_keras_serializable(package="Custom")
def dice_coef(y_true, y_pred, smooth=1e-6):
    y_true_f = tf.reshape(y_true, [-1])
    y_pred_f = tf.reshape(y_pred, [-1])
    intersection = tf.reduce_sum(y_true_f * y_pred_f)
    numerador = 2. * intersection + smooth
    denominador = tf.reduce_sum(y_true_f) + tf.reduce_sum(y_pred_f) + smooth
    return numerador / denominador

@keras.saving.register_keras_serializable(package="Custom")
def dice_loss(y_true, y_pred):
    return 1.0 - dice_coef(y_true, y_pred)

@keras.saving.register_keras_serializable(package="Custom")
def bce_dice_loss(y_true, y_pred):
    y_true_f = tf.reshape(y_true, [-1])
    y_pred_f = tf.reshape(y_pred, [-1])
    bce = tf.keras.losses.binary_crossentropy(y_true_f, y_pred_f)
    return tf.reduce_mean(bce) + dice_loss(y_true, y_pred)

In [4]:
# MODELO

def bloque_convolucion_doble(x, num_filtros):
    x = layers.Conv2D(num_filtros, kernel_size=3, padding="same", activation="relu", kernel_initializer="he_normal")(x)
    x = layers.Conv2D(num_filtros, kernel_size=3, padding="same", activation="relu", kernel_initializer="he_normal")(x)
    return x

def construir_unet(input_shape=(128, 128, 3)):
    inputs = layers.Input(shape=input_shape)
    
    # Encoder
    c1 = bloque_convolucion_doble(inputs, 16)
    p1 = layers.MaxPooling2D(pool_size=(2, 2))(c1)
    c2 = bloque_convolucion_doble(p1, 32)
    p2 = layers.MaxPooling2D(pool_size=(2, 2))(c2)
    c3 = bloque_convolucion_doble(p2, 64)
    p3 = layers.MaxPooling2D(pool_size=(2, 2))(c3)
    c4 = bloque_convolucion_doble(p3, 128)
    p4 = layers.MaxPooling2D(pool_size=(2, 2))(c4)
    
    # Bottleneck
    c5 = bloque_convolucion_doble(p4, 256)
    
    # Decoder
    u6 = layers.Conv2DTranspose(128, kernel_size=2, strides=(2, 2), padding="same")(c5)
    u6 = layers.concatenate([u6, c4])
    c6 = bloque_convolucion_doble(u6, 128)
    u7 = layers.Conv2DTranspose(64, kernel_size=2, strides=(2, 2), padding="same")(c6)
    u7 = layers.concatenate([u7, c3])
    c7 = bloque_convolucion_doble(u7, 64)
    u8 = layers.Conv2DTranspose(32, kernel_size=2, strides=(2, 2), padding="same")(c7)
    u8 = layers.concatenate([u8, c2])
    c8 = bloque_convolucion_doble(u8, 32)
    u9 = layers.Conv2DTranspose(16, kernel_size=2, strides=(2, 2), padding="same")(c8)
    u9 = layers.concatenate([u9, c1])
    c9 = bloque_convolucion_doble(u9, 16)
    
    outputs = layers.Conv2D(1, kernel_size=1, padding="same", activation="sigmoid")(c9)
    return models.Model(inputs=[inputs], outputs=[outputs], name="UNet_Segmentacion_Medica")

modelo_ejemplo = construir_unet()
modelo_ejemplo.summary()

Model: "UNet_Segmentacion_Medica"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                  ┃ Output Shape              ┃         Param # ┃ Connected to               ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)      │ (None, 128, 128, 3)       │               0 │ -                          │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ conv2d (Conv2D)               │ (None, 128, 128, 16)      │             448 │ input_layer[0][0]          │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ conv2d_1 (Conv2D)             │ (None, 128, 128, 16)      │           2,320 │ conv2d[0][0]               │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ max_pooling2d (MaxPooling2D)  │ (None, 64, 64, 16)        │               0 │ conv2d_1[0][0]             │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ conv2d_2 (Conv2D)             │ (None, 64, 64, 32)        │           4,640 │ max_pooling2d[0][0]        │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ conv2d_3 (Conv2D)             │ (None, 64, 64, 32)        │           9,248 │ conv2d_2[0][0]             │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ max_pooling2d_1               │ (None, 32, 32, 32)        │               0 │ conv2d_3[0][0]             │
│ (MaxPooling2D)                │                           │                 │                            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ conv2d_4 (Conv2D)             │ (None, 32, 32, 64)        │          18,496 │ max_pooling2d_1[0][0]      │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ conv2d_5 (Conv2D)             │ (None, 32, 32, 64)        │          36,928 │ conv2d_4[0][0]             │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ max_pooling2d_2               │ (None, 16, 16, 64)        │               0 │ conv2d_5[0][0]             │
│ (MaxPooling2D)                │                           │                 │                            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ conv2d_6 (Conv2D)             │ (None, 16, 16, 128)       │          73,856 │ max_pooling2d_2[0][0]      │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ conv2d_7 (Conv2D)             │ (None, 16, 16, 128)       │         147,584 │ conv2d_6[0][0]             │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ max_pooling2d_3               │ (None, 8, 8, 128)         │               0 │ conv2d_7[0][0]             │
│ (MaxPooling2D)                │                           │                 │                            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ conv2d_8 (Conv2D)             │ (None, 8, 8, 256)         │         295,168 │ max_pooling2d_3[0][0]      │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ conv2d_9 (Conv2D)             │ (None, 8, 8, 256)         │         590,080 │ conv2d_8[0][0]             │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ conv2d_transpose              │ (None, 16, 16, 128)       │         131,20

 Total params: 1,941,105 (7.40 MB)

 Trainable params: 1,941,105 (7.40 MB)

 Non-trainable params: 0 (0.00 B)

In [5]:
# DATA GENERATOR

class DataGenerator(keras.utils.PyDataset):
    def __init__(self, rutas_imagenes, rutas_gt, batch_size=4, patch_size=(128, 128), workers=1, use_multiprocessing=False):
        """
        Constructor del generador.
        
        NOTA SOBRE TERMINOLOGÍA:
        - 'gt' = Ground Truth (verdad fundamental)
        - Representa las segmentaciones manuales de referencia anotadas por expertos
        - En el dataset DRIVE se encuentran en las carpetas '1st_manual' y '2nd_manual'
        - Las carpetas 'mask' contienen solo máscaras del campo de visión (FoV), no segmentaciones
        
        Llamamos al constructor padre de PyDataset para habilitar el multiprocesamiento.
        """
        super().__init__(workers=workers, use_multiprocessing=use_multiprocessing)
        self.rutas_imagenes = rutas_imagenes
        self.rutas_gt = rutas_gt
        self.batch_size = batch_size
        self.patch_size = patch_size

    def __len__(self):
        return int(np.floor(len(self.rutas_imagenes) / self.batch_size))

    def __getitem__(self, index):
        
        # 1. Extraer las rutas específicas para este lote
        rutas_batch_x = self.rutas_imagenes[index * self.batch_size : (index + 1) * self.batch_size]
        rutas_batch_y = self.rutas_gt[index * self.batch_size : (index + 1) * self.batch_size]

        # 2. Inicializar matrices vacías para almacenar los parches recortados
        # X tendrá forma (batch_size, 128, 128, 3) -> 3 canales de color (RGB)
        # y tendrá forma (batch_size, 128, 128, 1) -> 1 canal (Blanco y negro con ground truth)
        X = np.empty((self.batch_size, *self.patch_size, 3), dtype=np.float32)
        y = np.empty((self.batch_size, *self.patch_size, 1), dtype=np.float32)

        # 3. Procesar cada imagen del lote actual
        for i, (ruta_img, ruta_gt) in enumerate(zip(rutas_batch_x, rutas_batch_y)):
            
            ojo_bgr = cv2.imread(ruta_img)
            if ojo_bgr is None:
                raise FileNotFoundError(f"No se pudo cargar la imagen: {ruta_img}. Verifica que la ruta existe.")
            
            ojo_rgb = cv2.cvtColor(ojo_bgr, cv2.COLOR_BGR2RGB)

            # FILTRO CLAHE
            # Pasamos a espacio de color LAB (Luminosidad, A, B)
            ojo_lab = cv2.cvtColor(ojo_rgb, cv2.COLOR_RGB2LAB)
            clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
            # Se lo aplicamos solo al canal de luminosidad (L) para no distorsionar colores
            ojo_lab[:, :, 0] = clahe.apply(ojo_lab[:, :, 0])
            ojo_rgb = cv2.cvtColor(ojo_lab, cv2.COLOR_LAB2RGB)

            # Para el ground truth, se lee en escala de grises (cv2.IMREAD_GRAYSCALE).
            gt_array = cv2.imread(ruta_gt, cv2.IMREAD_GRAYSCALE)
            if gt_array is None:
                raise FileNotFoundError(f"No se pudo cargar el ground truth: {ruta_gt}. Verifica que la ruta existe.")

            
            img_normalizada = ojo_rgb / 255.0
            gt_normalizada = gt_array / 255.0

            # Parcheado 
            # Generar unas coordenadas X e Y aleatorias.
            # Asegurarse de que las coordenadas no se salgan de los bordes de la imagen original al sumarle los 128 píxeles del patch_size.
            # Recortar EXACTAMENTE las mismas coordenadas tanto en 'img' como en 'gt'.

            alto_original, ancho_original = img_normalizada.shape[:2]
            limite_y = alto_original - self.patch_size[0]
            limite_x = ancho_original - self.patch_size[1]

            x_random = random.randint(0, limite_x)
            y_random = random.randint(0, limite_y)

            parche_img = img_normalizada[y_random : y_random + self.patch_size[0], x_random : x_random + self.patch_size[1]]
            parche_gt = gt_normalizada[y_random : y_random + self.patch_size[0], x_random : x_random + self.patch_size[1]]

            # Data Augmentation 
            # Volteo horizontal
            
            if random.random() > 0.5: 
                parche_img = cv2.flip(parche_img, 1)  # 1 para flip horizontal
                parche_gt = cv2.flip(parche_gt, 1)

            # Volteo vertical
            if random.random() > 0.5:
                parche_img = cv2.flip(parche_img, 0)
                parche_gt = cv2.flip(parche_gt, 0)

            # Rotaciones aleatorias de 90, 180 o 270 grados
            if random.random() > 0.5:
                k = random.choice([0, 1, 2])  # 0=90°, 1=180°, 2=270°
                parche_img = cv2.rotate(parche_img, k)
                parche_gt = cv2.rotate(parche_gt, k)
            
            # Alterción aleatoria del brillo (solo en la imagen, no en el ground truth)
            if random.random() > 0.5:
                factor_brillo = random.uniform(0.8, 1.2)  # Brillo entre 80% y 120%
                parche_img = np.clip(parche_img * factor_brillo, 0.0, 1.0)  # Asegura que los valores sigan entre 0 y 1

            # Ruido Gausiano (simula mala calidad de cámara)
            if random.random() > 0.5:
                ruido = np.random.normal(0, 0.05, parche_img.shape)  # Media=0, Desviación=0.05
                parche_img = np.clip(parche_img + ruido, 0.0, 1.0)
            
            # Guardar los parches finales en las matrices del batch
            X[i,] = parche_img
            
            y[i,] = np.expand_dims(parche_gt, axis=-1)

        return X, y

In [6]:
# ENTRENAMIENTO
# NO EJECUTAR SI LOS MODELOS YA ESTÁN CARGADOS

img_dir = os.path.join("data", "training", "images")
gt_dir = os.path.join("data", "training", "1st_manual")
models_dir = os.path.join("models")

os.makedirs(models_dir, exist_ok=True)

batch_size = 4
patch_size = (128, 128)
epochs = 200
n_folds = 5
learning_rate = 0.0001
patience = 25
input_shape = (128, 128, 3)
reps = 10
seed = 42

np.random.seed(seed)
tf.random.set_seed(seed)

def get_paths(dir_img, dir_gt):
    exts = ["*.tif", "*.tiff", "*.png", "*.jpg", "*.jpeg", "*.gif", "*.bmp"]
    imgs = []
    gts = []

    for ext in exts:
        imgs += glob.glob(os.path.join(dir_img, ext))
        gts += glob.glob(os.path.join(dir_gt, ext))

    imgs = sorted(set(imgs))
    gts = sorted(set(gts))

    if not imgs or not gts or len(imgs) != len(gts):
        raise ValueError("Error cargando los archivos. Revisa las carpetas y que haya el mismo número de imágenes y máscaras.")

    print(f"Cargados {len(imgs)} pares de imágenes.")
    return np.array(imgs), np.array(gts)

def get_callbacks(fold):
    ruta_modelo = os.path.join(models_dir, f"fold_{fold}.keras")
    
    return [
        EarlyStopping(monitor="val_dice_coef", mode="max", patience=patience, restore_best_weights=True, verbose=1),
        ModelCheckpoint(filepath=ruta_modelo, monitor="val_dice_coef", mode="max", save_best_only=True, verbose=1),
        ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=7, min_lr=1e-7, verbose=1)
    ]

def train(imgs, gts):
    kf = KFold(n_splits=n_folds, shuffle=True, random_state=seed)
    resultados = []

    for fold, (train_idx, val_idx) in enumerate(kf.split(imgs), 1):
        print(f"\n--- Empezando fold {fold}/{n_folds} ---")
        
        gen_train = DataGenerator(
            rutas_imagenes=list(imgs[train_idx]) * reps,
            rutas_gt=list(gts[train_idx]) * reps,
            batch_size=batch_size,
            patch_size=patch_size
        )
        
        gen_val = DataGenerator(
            rutas_imagenes=list(imgs[val_idx]),
            rutas_gt=list(gts[val_idx]),
            batch_size=batch_size,
            patch_size=patch_size
        )

        modelo = construir_unet(input_shape=input_shape)
        modelo.compile(
            optimizer=Adam(learning_rate=learning_rate),
            loss=bce_dice_loss,
            metrics=[dice_coef]
        )

        historial = modelo.fit(
            gen_train,
            validation_data=gen_val,
            epochs=epochs,
            callbacks=get_callbacks(fold),
            verbose=1
        )

        mejor_val = max(historial.history["val_dice_coef"])
        resultados.append(mejor_val)
        print(f"Fold {fold} terminado con DICE de {mejor_val:.4f}")

    print("\nResumen final:")
    for i, res in enumerate(resultados, 1):
        print(f"Fold {i}: {res:.4f}")
    print(f"Media: {np.mean(resultados):.4f} | Desviación: {np.std(resultados):.4f}")


if __name__ == "__main__":
    print("Arrancando script...")
    imgs, gts = get_paths(img_dir, gt_dir)

    print("\nComprobando que emparejan bien:")
    for i, g in zip(imgs[:3], gts[:3]):
        print(f" - {os.path.basename(i)} -> {os.path.basename(g)}")

    #train(imgs, gts) # Comentar para que no se ejecute el entrenamiento si los modelos ya se encuentran cargados

Arrancando script...
Cargados 20 pares de imágenes.

Comprobando que emparejan bien:
 - 21_training.tif -> 21_manual1.gif
 - 22_training.tif -> 22_manual1.gif
 - 23_training.tif -> 23_manual1.gif


In [10]:
# EVALUACIÓN DE LAS PREDICCIONES


if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")

src_path = os.path.join(os.getcwd(), "src")
if src_path not in sys.path:
    sys.path.insert(0, src_path)

img_dir = os.path.join("data", "test", "images")
gt1_dir = os.path.join("data", "test", "1st_manual")
gt2_dir = os.path.join("data", "test", "2nd_manual")
fov_dir = os.path.join("data", "test", "mask")
models_dir = os.path.join("models")
out_dir = os.path.join("predictions")

os.makedirs(out_dir, exist_ok=True)

patch_size = (128, 128)
input_shape = (128, 128, 3)

def leer_img(ruta):
    img = cv2.imread(ruta, cv2.IMREAD_COLOR)
    if img is None:
        raise FileNotFoundError(f"Error al leer la imagen: {ruta}")
    
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    
    # Aplicar filtro CLAHE al canal L
    img_lab = cv2.cvtColor(img, cv2.COLOR_RGB2LAB)
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    img_lab[:, :, 0] = clahe.apply(img_lab[:, :, 0])
    img = cv2.cvtColor(img_lab, cv2.COLOR_LAB2RGB)
    
    return img.astype(np.float32) / 255.0

def leer_mascara(ruta):
    mascara = cv2.imread(ruta, cv2.IMREAD_GRAYSCALE)
    if mascara is None:
        return None
    return (mascara > 127).astype(np.float32)

def obtener_numero(nombre):
    num = re.match(r"(\d+)", nombre)
    return num.group(1) if num else nombre

def predecir_parches(modelo, img, p_size=(128, 128), stride=64):
    h, w, _ = img.shape
    ph, pw = p_size

    acumulador = np.zeros((h, w), dtype=np.float32)
    contador = np.zeros((h, w), dtype=np.float32)

    filas = list(range(0, h - ph + 1, stride))
    cols = list(range(0, w - pw + 1, stride))

    if not filas: filas = [0]
    if not cols: cols = [0]
    if filas[-1] + ph < h: filas.append(h - ph)
    if cols[-1] + pw < w: cols.append(w - pw)

    parches = []
    coords = []
    
    for r in filas:
        for c in cols:
            parches.append(img[r:r+ph, c:c+pw, :])
            coords.append((r, c))

    parches = np.array(parches, dtype=np.float32)
    preds = modelo.predict(parches, verbose=0, batch_size=16)

    for i, (r, c) in enumerate(coords):
        acumulador[r:r+ph, c:c+pw] += preds[i, :, :, 0]
        contador[r:r+ph, c:c+pw] += 1.0

    contador[contador == 0] = 1.0
    return acumulador / contador

def cargar_modelos(carpeta):
    rutas = sorted(glob.glob(os.path.join(carpeta, "fold_*.keras")))
    if not rutas:
        raise FileNotFoundError("No se han encontrado modelos. Ejecuta train.py primero.")
    
    modelos = []
    for r in rutas:
        print(f"Cargando modelo: {r}")
        m = keras.models.load_model(r, custom_objects={"dice_coef": dice_coef, "dice_loss": dice_loss})
        modelos.append(m)
        
    return modelos

def ensemble_predict(modelos, img, p_size=(128, 128), stride=64):
    suma = np.zeros(img.shape[:2], dtype=np.float32)
    
    img_h = cv2.flip(img, 1)
    img_v = cv2.flip(img, 0)
    
    for m in modelos:
        suma += predecir_parches(m, img, p_size, stride)
        
        p_h = predecir_parches(m, img_h, p_size, stride)
        suma += cv2.flip(p_h, 1)
        
        p_v = predecir_parches(m, img_v, p_size, stride)
        suma += cv2.flip(p_v, 0)
        
    return suma / (len(modelos) * 3.0)

def obtener_rutas_test():
    exts = ["*.tif", "*.png", "*.jpg", "*.bmp", "*.gif"]
    imgs, gt1s, gt2s, fovs = [], [], [], []

    for ext in exts:
        # Aquí está la corrección: usar os.path.join para cada búsqueda
        imgs += glob.glob(os.path.join(img_dir, ext))
        gt1s += glob.glob(os.path.join(gt1_dir, ext))
        gt2s += glob.glob(os.path.join(gt2_dir, ext))
        fovs += glob.glob(os.path.join(fov_dir, ext))

    idx_imgs = {obtener_numero(os.path.basename(r)): r for r in imgs}
    idx_gt1 = {obtener_numero(os.path.basename(r)): r for r in gt1s}
    idx_gt2 = {obtener_numero(os.path.basename(r)): r for r in gt2s}
    idx_fov = {obtener_numero(os.path.basename(r)): r for r in fovs}

    comunes = sorted(set(idx_imgs) & set(idx_gt1) & set(idx_gt2))
    
    if not comunes:
        raise FileNotFoundError("No se encontraron coincidencias de archivos en las carpetas de test.")

    imgs_fin = [idx_imgs[n] for n in comunes]
    gt1_fin = [idx_gt1[n] for n in comunes]
    gt2_fin = [idx_gt2[n] for n in comunes]
    fov_fin = [idx_fov.get(n) for n in comunes]

    print(f"Encontradas {len(comunes)} imágenes para evaluar.")
    return imgs_fin, gt1_fin, gt2_fin, fov_fin

def calcular_dice(pred, mask):
    inter = np.sum(pred * mask)
    denominador = np.sum(pred) + np.sum(mask)
    if denominador == 0:
        return 1.0
    return (2.0 * inter + 1e-6) / (denominador + 1e-6)

def run_evaluation():
    modelos = cargar_modelos(models_dir)
    imgs, gt1s, gt2s, fovs = obtener_rutas_test()

    res_dice_prom = []
    res_dice1, res_dice2 = [], []
    res_precision, res_recall = [], []

    for i, (r_img, r_gt1, r_gt2, r_fov) in enumerate(zip(imgs, gt1s, gt2s, fovs), 1):
        nombre = os.path.basename(r_img).split('.')[0]
        print(f"\n[{i}/{len(imgs)}] Evaluando: {nombre}")

        img = leer_img(r_img)
        gt1 = leer_mascara(r_gt1)
        gt2 = leer_mascara(r_gt2)
        fov = leer_mascara(r_fov) if r_fov else None

        prob = ensemble_predict(modelos, img, patch_size, 64)

        prob_uint8 = (prob * 255).astype(np.uint8)
        
        
        umbral_otsu_val, _ = cv2.threshold(prob_uint8, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
        umbral_real = umbral_otsu_val / 255.0
        
        umbral_ajustado = max(0, umbral_real - 0.040)
        _, pred_uint8 = cv2.threshold(prob_uint8, int(umbral_ajustado * 255), 255, cv2.THRESH_BINARY)
        print(f"    -> Umbral Otsu: {umbral_real:.4f} | Ajustado: {umbral_ajustado:.4f}")
        
        pred = (pred_uint8 / 255.0).astype(np.float32)
        
        kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (3, 3))
        pred_uint8 = cv2.morphologyEx(pred_uint8, cv2.MORPH_CLOSE, kernel, iterations=1)
        pred = (pred_uint8 / 255.0).astype(np.float32)

        # Filtro de limpiado
        num_labels, labels, stats, centroids = cv2.connectedComponentsWithStats(pred_uint8, connectivity=8)
        pred_limpia = np.zeros_like(pred)
        for j in range(1, num_labels):
            if stats[j, cv2.CC_STAT_AREA] >= 2:  
                pred_limpia[labels == j] = 1.0
        pred = pred_limpia

        if fov is not None:
            if fov.shape != pred.shape:
                fov = cv2.resize(fov, (pred.shape[1], pred.shape[0]), interpolation=cv2.INTER_NEAREST)
            pred *= fov
            gt1 *= fov
            gt2 *= fov

        d1 = calcular_dice(pred, gt1)
        d2 = calcular_dice(pred, gt2)
        d_prom = (d1 + d2) / 2.0
        
        res_dice1.append(d1)
        res_dice2.append(d2)
        res_dice_prom.append(d_prom)

        gt_union = np.maximum(gt1, gt2)
        gt_plano = gt_union.flatten()
        pred_plano = pred.flatten()
            
        prec = precision_score(gt_plano, pred_plano, zero_division=0)
        rec = recall_score(gt_plano, pred_plano, zero_division=0)
        
        res_precision.append(prec)
        res_recall.append(rec)

        print(f"  Exp1: {d1:.4f} | Exp2: {d2:.4f} | Media: {d_prom:.4f} | Prec: {prec:.4f} | Rec: {rec:.4f}")

        cv2.imwrite(os.path.join(out_dir, f"{nombre}_pred.png"), (pred * 255).astype(np.uint8))

        fig, axes = plt.subplots(2, 2, figsize=(10, 10))
        fig.suptitle(f"{nombre} - DICE Media: {d_prom:.4f}")
        
        axes[0, 0].imshow(img)
        axes[0, 0].set_title("Original")
        axes[0, 0].axis("off")
        
        axes[0, 1].imshow(gt1, cmap="gray")
        axes[0, 1].set_title("Experto 1")
        axes[0, 1].axis("off")
        
        axes[1, 0].imshow(gt2, cmap="gray")
        axes[1, 0].set_title("Experto 2")
        axes[1, 0].axis("off")
        
        axes[1, 1].imshow(pred, cmap="gray")
        axes[1, 1].set_title(f"Predicción (umbral {umbral_real:.4f})")
        axes[1, 1].axis("off")
        
        plt.tight_layout()
        plt.savefig(os.path.join(out_dir, f"{nombre}_comparacion.png"), dpi=100, bbox_inches="tight")
        plt.close(fig)

    print("\n--- RESULTADOS FINALES ---")
    nombres = [os.path.basename(r).split('.')[0] for r in imgs]
    
    for nom, d1, d2, d_prom in zip(nombres, res_dice1, res_dice2, res_dice_prom):
        estado = "OK" if d_prom >= 0.75 else "FAIL"
        print(f"[{estado}] {nom} -> DICE Exp1: {d1:.4f} | DICE Exp2: {d2:.4f} | DICE Media: {d_prom:.4f}")
        
    media_total = np.mean(res_dice_prom)
    aprobados = sum(1 for d in res_dice_prom if d >= 0.75)
    
    print("\nResumen general:")
    print(f"  DICE Experto 1 : {np.mean(res_dice1):.4f}")
    print(f"  DICE Experto 2 : {np.mean(res_dice2):.4f}")
    print(f"  DICE Media     : {media_total:.4f}")
    print(f"  Precisión      : {np.mean(res_precision):.4f}")
    print(f"  Recall         : {np.mean(res_recall):.4f}")
    print(f"  Aprobados      : {aprobados}/{len(res_dice_prom)}")
    
    if media_total >= 0.75:
        print("\n¡Umbral de 0.75 superado!")
    else:
        print(f"\nFaltan {0.75 - media_total:.4f} puntos para superar el umbral.")


if __name__ == "__main__":
    run_evaluation()

Cargando modelo: models\fold_1.keras
Cargando modelo: models\fold_2.keras
Cargando modelo: models\fold_3.keras
Cargando modelo: models\fold_4.keras
Cargando modelo: models\fold_5.keras
Encontradas 20 imágenes para evaluar.

[1/20] Evaluando: 01_test
    -> Umbral Otsu: 0.4314 | Ajustado: 0.3914
  Exp1: 0.7801 | Exp2: 0.7898 | Media: 0.7850 | Prec: 0.8266 | Rec: 0.7925

[2/20] Evaluando: 02_test
    -> Umbral Otsu: 0.4431 | Ajustado: 0.4031
  Exp1: 0.8244 | Exp2: 0.8307 | Media: 0.8276 | Prec: 0.9147 | Rec: 0.7467

[3/20] Evaluando: 03_test
    -> Umbral Otsu: 0.4157 | Ajustado: 0.3757
  Exp1: 0.7676 | Exp2: 0.7924 | Media: 0.7800 | Prec: 0.8949 | Rec: 0.6793

[4/20] Evaluando: 04_test
    -> Umbral Otsu: 0.4431 | Ajustado: 0.4031
  Exp1: 0.7912 | Exp2: 0.7969 | Media: 0.7940 | Prec: 0.8908 | Rec: 0.7020

[5/20] Evaluando: 05_test
    -> Umbral Otsu: 0.4275 | Ajustado: 0.3875
  Exp1: 0.7847 | Exp2: 0.8094 | Media: 0.7970 | Prec: 0.9203 | Rec: 0.6762

[6/20] Evaluando: 06_test
    -> Umb